# 04 — RAG Pipeline Demo

End-to-end demonstration of the full pipeline:
`user query → semantic retrieval → nutritional re-ranking → LLM generation → explainable response`

Requires `GEMINI_API_KEY` in `.env` (or `OPENAI_API_KEY` if using OpenAI).

In [4]:
import os
os.chdir('..')  # run from project root so all data/ paths resolve correctly

import sys
sys.path.insert(0, '.')
from dotenv import load_dotenv
load_dotenv('.env')

False

In [2]:
# --- Retriever-only demo (no API key needed) ---
from src.embeddings.retriever import RecipeRetriever

retriever = RecipeRetriever(config_path='configs/config.yaml')

test_queries = [
    'diabetic friendly low sugar dinner',
    'high protein gluten free post workout meal',
    'low fat heart healthy soup',
]

for q in test_queries:
    results = retriever.retrieve(q, top_k=3)
    print(f'\nQuery: "{q}"')
    for _, row in results.iterrows():
        flags = [c.replace('is_', '') for c in results.columns if c.startswith('is_') and row[c]]
        print(f"  → {row['name'].title()[:50]:<50} | hybrid={row['hybrid_score']:.3f} | {flags}")

/Users/advahelman/RAG/venv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange



Query: "diabetic friendly low sugar dinner"
  → Absolutely Sugar Free Frosting                     | hybrid=0.754 | ['diabetic_friendly', 'gluten_free', 'low_sodium', 'low_calorie', 'keto_friendly', 'low_carb', 'low_sugar']
  → Barbecued Fish Steaks                              | hybrid=0.725 | ['diabetic_friendly', 'gluten_free', 'high_protein', 'low_calorie', 'low_carb', 'low_sugar']
  → Sugar Free Punch                                   | hybrid=0.721 | ['diabetic_friendly', 'gluten_free', 'low_fat', 'low_sodium', 'low_calorie', 'low_carb', 'low_sugar']

Query: "high protein gluten free post workout meal"
  → Homemade Pork And Beans  Slow Cook Method  Gf      | hybrid=0.706 | ['gluten_free', 'high_protein']
  → High Protein Greek Chicken Salad                   | hybrid=0.705 | ['gluten_free', 'high_protein', 'low_calorie', 'low_carb']
  → Gluten Free Breaded Chicken                        | hybrid=0.704 | ['diabetic_friendly', 'gluten_free', 'high_protein', 'low_calorie', 'low_car

In [3]:
from src.rag.generator import RAGPipeline

pipeline = RAGPipeline(config_path='configs/config.yaml')

## Demo 1: Diabetic-Friendly Dinner

In [4]:
result = pipeline.recommend('I need a diabetic friendly dinner that is also filling')

print('QUERY:', result['query'])
print('DETECTED CONSTRAINTS:', result['dietary_constraints'])
print()
print('TOP RETRIEVED RECIPES:')
for r in result['recipes']:
    print(f"  {r['name'].title()[:50]:<50} | hybrid={r['hybrid_score']:.3f} | nutri={r['nutritional_score']:.2f}")
print()
print('LLM RESPONSE:')
print('-' * 60)
print(result['response'])

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 25.529066927s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.goog

## Demo 2: High-Protein Gluten-Free Post-Workout

In [ ]:
result = pipeline.recommend('high protein gluten free meal for after the gym')
print('DETECTED CONSTRAINTS:', result['dietary_constraints'])
print()
print('EXPLANATION FOR TOP RESULT:')
print(result['recipes'][0]['explanation'])
print()
print('LLM RESPONSE:')
print(result['response'])

## Demo 3: Multi-Constraint Query

Tests the system's ability to handle multiple simultaneous constraints.

In [5]:
result = pipeline.recommend('I have diabetes and gluten intolerance, need a high protein lunch')
print('DETECTED CONSTRAINTS:', result['dietary_constraints'])
print('Results after filtering:', len(result['recipes']))
print()
print('LLM RESPONSE:')
print(result['response'])

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 21.313764553s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.goog

## Nutritional Score Breakdown

Show the rule-based scores and explanations for the top result.

In [ ]:
from src.scoring.nutritional_scorer import score_recipe

top_recipe = result['recipes'][0]
constraints = result['dietary_constraints']

print(f"Recipe: {top_recipe['name'].title()}")
print(f"Constraints: {constraints}")
print()

score, explanations = score_recipe(top_recipe, constraints)
print(f'Nutritional Score: {score:.2f}/1.00')
print('Why it qualifies:')
for exp in explanations:
    print(f'  • {exp}')